# OpenCV Color Recognition

In this tutorial, we'll integrate some functions to modify frame images within OpenCV, such as blurring, color space conversion, erosion, and dilation.

## Preparation
Since the product automatically runs the main program on startup, and the main program occupies the camera resources, this tutorial cannot be used in that state. You need to stop the main program or disable its automatic startup before restarting the robot.

It should be noted that because the robot's main program uses multithreading and is configured to start automatically via a service, the usual method of `sudo killall python` often does not work. Therefore, we will introduce a method to disable the main program's automatic startup.

If you have already disabled the automatic startup of the robot's main program, you do not need to perform the following "Stopping the Main Program" section.

### Stopping the Main Program
1. Click the "" icon next to the tab at the top of this page to open a new tab named Launcher.
2. Click Terminal under Other to open the terminal window.
3. In the terminal window, type `bash` and press Enter.
4. You can now use the Bash Shell to control the robot.
5. Enter the command: `systemctl --user stop ugv-app.service`.
6. On the terminal page, after the command has been executed, continue with the remaining part of this tutorial.

## Example

The following code block can be run directly:

1. Select the code block below.
2. Press Shift + Enter to run the code block.
3. Watch the real-time video window.
4. Press STOP to close the real-time video and release the camera resources.

### If you cannot see the real-time camera feed when running:

- Click on Kernel -> Shut down all kernels above.
- Close the current section tab and open it again.
- Click `STOP` to release the camera resources, then run the code block again.
- Reboot the device.

### Execution

By default, we detect green balls in the example. Ensure that there are no green objects in the background to avoid interfering with the color recognition function. You can also modify the detection color (in the LAB color space) through secondary development.

In [ ]:
import cv2
import imutils
from picamera2 import Picamera2  # Library for accessing Raspberry Pi Camera
import numpy as np  # Library for mathematical calculations
from IPython.display import display, Image  # Library for displaying images in Jupyter Notebook
import ipywidgets as widgets  # Library for creating interactive widgets such as buttons
import threading  # Library for creating new threads to execute tasks asynchronously

# Create a "Stop" button that users can click to stop the video stream
# ================================================================
stopButton = widgets.ToggleButton(
    value=False,
    description='Stop',
    disabled=False,
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Description',
    icon='square' # (FontAwesome names without the `fa-` prefix)
)

# Define the display function to process video frames and recognize objects of specific colors
def view(button):
    # If you are using a CSI camera, uncomment the picam2 related code below, 
    # and comment out the camera related code.
    # This is because the latest version of OpenCV (4.9.0.80) no longer supports CSI cameras, 
    # and you need to use picamera2 to capture camera images.
    
    # picam2 = Picamera2()  # Create an instance of Picamera2
    # picam2.configure(picam2.create_video_configuration(main={"format": 'XRGB8888', "size": (640, 480)}))  # Configure camera parameters
    # picam2.start()  # Start the camera
    
    camera = cv2.VideoCapture(0) 
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    display_handle=display(None, display_id=True)  # Create a display handle to update displayed images
    i = 0
    
    # Define the color range to be detected
    color_upper = np.array([255, 110, 195])
    color_lower = np.array([0, 0, 0])
    
    while True:
        # img = picam2.capture_array() # Capture a frame from the camera
        _, img = camera.read()
        # frame = cv2.flip(frame, 1) # if your camera reverses your image

        # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        cx, cy, w = None, None, None
        img_h, img_w = img.shape[:2]
        overlay_buffer = np.zeros_like(img)
        center_x, center_y = img_w // 2, img_h // 2
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        mask = cv2.inRange(lab, color_lower, color_upper)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            c = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(c)
            if area > 100:  
                ((x, y), radius) = cv2.minEnclosingCircle(c)
                circularity = area / (math.pi * radius * radius)
                if 0.5 < circularity < 1.3:  
                    cx, cy, w = int(x), int(y), int(radius*2)

                    cv2.circle(img, (cx, cy), int(radius), (0, 255, 0), 2)
                    cv2.circle(img, (cx, cy), 3, (255, 0, 0), -1)
                    # print(f'Tracking ball at ({cx}, {cy}), area={area:.1f}, circularity={circularity:.2f}')

        _, frame = cv2.imencode('.jpeg', img)  # Encode the frame to JPEG format
        display_handle.update(Image(data=frame.tobytes()))  # Update the displayed image
        if stopButton.value==True:  # Check if the "Stop" button has been pressed
            # picam2.close()  # If so, close the camera
            camera.release() # If yes, close the camera
            display_handle.update(None)  # Clear the displayed content


# Display the "Stop" button and start a thread to execute the display function
# ================================================================
display(stopButton)
thread = threading.Thread(target=view, args=(stopButton,))
thread.start()